# 面试问题：Agent 需要执行代码时怎样做沙箱、资源限制和结果审计？

**一句话回答**：不要在应用进程中 `eval/exec` 模型文本。把代码作为不可信 workload，在独立身份、隔离文件系统/网络和 CPU/内存/时间/输出配额中执行；只挂载必要输入，副作用写 staging，完成后审计 diff 再提交。教学实现可用白名单 AST 解释器证明 capability/gas 思路，但它不是操作系统沙箱。

本 Notebook 不执行任意 Python，而是手写安全算术 AST VM、gas/depth/数值/输出限制、虚拟只读文件能力和 staged writes。

In [ ]:
import ast,hashlib,json,math,operator  # 导入本单元所需的依赖。

SEED124=12401  # 计算并保存当前步骤的中间状态。
assert SEED124==12401  # 用受控断言验证关键不变量。
assert math.isfinite(SEED124)  # 用受控断言验证关键不变量。
assert ast.parse("1+2",mode="eval").body.__class__ is ast.BinOp  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"untrusted-code").hexdigest()!=hashlib.sha256(b"trusted-host").hexdigest()  # 用受控断言验证关键不变量。

## 1. 威胁模型覆盖代码、依赖、数据和输出

风险包括任意系统调用、网络外传、读取凭证、fork bomb、无限循环、大整数/压缩炸弹、依赖供应链和终端/HTML 输出注入。语言级过滤不是完整隔离；生产需要容器/microVM、低权限 UID、seccomp、只读根、网络默认拒绝和外层 watchdog。

In [ ]:
THREATS124={"filesystem","network","process","cpu","memory","output","secrets","supply_chain"}; CONTROLS124={"filesystem":"readonly_mount","network":"deny","process":"syscall_filter","cpu":"quota","memory":"quota","output":"cap","secrets":"none_mounted","supply_chain":"pinned_image"}  # 计算并保存当前步骤的中间状态。
assert set(CONTROLS124)==THREATS124  # 用受控断言验证关键不变量。
assert CONTROLS124["network"]=="deny"  # 用受控断言验证关键不变量。
assert CONTROLS124["secrets"]=="none_mounted"  # 用受控断言验证关键不变量。

## 2. 白名单 AST 只接受明确表达式子集

允许常量、变量、有限算术和少数纯函数；拒绝 attribute、subscript、lambda、comprehension、import 和任意函数调用。parser 成功不等于允许执行，必须递归检查每个 node。下面是 capability VM，不调用 Python `eval`。

In [ ]:
ALLOWED_BIN124={ast.Add:operator.add,ast.Sub:operator.sub,ast.Mult:operator.mul,ast.Div:operator.truediv}; ALLOWED_UNARY124={ast.UAdd:operator.pos,ast.USub:operator.neg}; PURE124={"abs":abs,"min":min,"max":max,"round":round}  # 计算并保存当前步骤的中间状态。
def validate_ast124(node,depth=0,max_depth=12):  # 定义本节可复用的核心函数。
    if depth>max_depth: raise ValueError("ast_depth")  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.Expression): return validate_ast124(node.body,depth+1,max_depth)  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.Constant) and type(node.value) in {int,float}: return  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.Name): return  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.BinOp) and type(node.op) in ALLOWED_BIN124: validate_ast124(node.left,depth+1,max_depth); validate_ast124(node.right,depth+1,max_depth); return  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.UnaryOp) and type(node.op) in ALLOWED_UNARY124: validate_ast124(node.operand,depth+1,max_depth); return  # 按当前条件选择后续控制路径。
    if isinstance(node,ast.Call) and isinstance(node.func,ast.Name) and node.func.id in PURE124 and not node.keywords: [validate_ast124(a,depth+1,max_depth) for a in node.args]; return  # 按当前条件选择后续控制路径。
    raise ValueError("forbidden_ast")  # 遇到非法合同立即显式失败。
validate_ast124(ast.parse("max(1,x*2)",mode="eval"))  # 计算并保存当前步骤的中间状态。
assert True  # 用受控断言验证关键不变量。
for bad in ("__import__('os')","x.__class__","[x for x in [1]]"):  # 遍历输入元素以累积或检查结果。
    try: validate_ast124(ast.parse(bad,mode="eval")); raise AssertionError("unsafe AST accepted")  # 尝试执行可能失败的受控操作。
    except ValueError as e: assert str(e)=="forbidden_ast"  # 捕获预期异常并验证失败分支。

## 3. 自定义解释器按节点消耗 gas

gas 防止超大 AST 计算，数值上限防止极端中间值。变量来自显式环境且仅允许数值；除零和未知变量作为结构化错误。即使纯表达式也需限制，因为大整数和深嵌套会耗尽资源。

In [ ]:
class VM124:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,env,gas=100,max_abs=1e9): self.env=dict(env); self.gas=gas; self.max_abs=max_abs  # 定义本节可复用的核心函数。
    def spend(self):  # 定义本节可复用的核心函数。
        self.gas-=1  # 计算并保存当前步骤的中间状态。
        if self.gas<0: raise ValueError("gas_exhausted")  # 按当前条件选择后续控制路径。
    def run(self,node):  # 定义本节可复用的核心函数。
        self.spend()  # 执行当前语句以推进本节示例。
        if isinstance(node,ast.Expression): return self.run(node.body)  # 按当前条件选择后续控制路径。
        if isinstance(node,ast.Constant): value=node.value  # 按当前条件选择后续控制路径。
        elif isinstance(node,ast.Name):  # 按当前条件选择后续控制路径。
            if node.id not in self.env or type(self.env[node.id]) not in {int,float}: raise ValueError("unknown_variable")  # 按当前条件选择后续控制路径。
            value=self.env[node.id]  # 计算并保存当前步骤的中间状态。
        elif isinstance(node,ast.BinOp): value=ALLOWED_BIN124[type(node.op)](self.run(node.left),self.run(node.right))  # 按当前条件选择后续控制路径。
        elif isinstance(node,ast.UnaryOp): value=ALLOWED_UNARY124[type(node.op)](self.run(node.operand))  # 按当前条件选择后续控制路径。
        elif isinstance(node,ast.Call): value=PURE124[node.func.id](*[self.run(a) for a in node.args])  # 按当前条件选择后续控制路径。
        else: raise ValueError("forbidden_ast")  # 执行当前语句以推进本节示例。
        if not math.isfinite(value) or abs(value)>self.max_abs: raise ValueError("numeric_limit")  # 按当前条件选择后续控制路径。
        return value  # 返回当前分支计算出的结果。
tree124=ast.parse("max(1,x*2)+3",mode="eval"); validate_ast124(tree124); vm124=VM124({"x":4}); result124=vm124.run(tree124)  # 计算并保存当前步骤的中间状态。
assert result124==11 and vm124.gas<100  # 用受控断言验证关键不变量。
try: VM124({},gas=1).run(tree124); raise AssertionError("gas ignored")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="gas_exhausted"  # 捕获预期异常并验证失败分支。
try: VM124({"x":1e9}).run(ast.parse("x*2",mode="eval")); raise AssertionError("numeric cap ignored")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="numeric_limit"  # 捕获预期异常并验证失败分支。

## 4. 文件能力使用虚拟句柄和路径 allowlist

不把宿主路径、HOME 或凭证暴露给 workload。输入按内容寻址挂载为只读 artifact，路径规范化后必须位于工作区；符号链接也需在宿主解析。下面 VM 只接收 artifact ID，不接收任意路径。

In [ ]:
artifacts124={"input-1":{"content":"1,2,3","tenant":"T1","readonly":True},"secret":{"content":"token","tenant":"T2","readonly":True}}  # 计算并保存当前步骤的中间状态。
def read_artifact124(artifact_id,tenant,max_bytes=100):  # 定义本节可复用的核心函数。
    a=artifacts124.get(artifact_id)  # 计算并保存当前步骤的中间状态。
    if not a or a["tenant"]!=tenant: raise ValueError("artifact_acl")  # 按当前条件选择后续控制路径。
    if len(a["content"].encode())>max_bytes: raise ValueError("artifact_size")  # 按当前条件选择后续控制路径。
    return a["content"]  # 返回当前分支计算出的结果。
assert read_artifact124("input-1","T1")=="1,2,3"  # 用受控断言验证关键不变量。
try: read_artifact124("secret","T1"); raise AssertionError("cross tenant read")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="artifact_acl"  # 捕获预期异常并验证失败分支。
assert artifacts124["input-1"]["readonly"]  # 用受控断言验证关键不变量。

## 5. 网络默认拒绝，代理请求经过宿主 broker

沙箱本身无网络；若任务需要下载，宿主提供窄 fetch capability，校验 scheme/host/IP、响应类型/大小和租户策略，并把响应作为不可信 artifact。DNS rebinding、云 metadata 和重定向链都要检查。

In [ ]:
from urllib.parse import urlparse  # 导入本单元所需的依赖。
ALLOWED_HOSTS124={"data.example.com"}  # 计算并保存当前步骤的中间状态。
def authorize_fetch124(url):  # 定义本节可复用的核心函数。
    p=urlparse(url); return p.scheme=="https" and p.hostname in ALLOWED_HOSTS124 and not p.username and p.port in {None,443}  # 计算并保存当前步骤的中间状态。
assert authorize_fetch124("https://data.example.com/file")  # 用受控断言验证关键不变量。
assert not authorize_fetch124("http://data.example.com/file")  # 用受控断言验证关键不变量。
assert not authorize_fetch124("https://user@data.example.com/file") and not authorize_fetch124("https://169.254.169.254/latest")  # 用受控断言验证关键不变量。

## 6. 输出和日志也设置 byte/line/结构上限

workload 可打印无限数据或控制字符攻击终端。采集器按 bytes 截断、标记 truncated，结构化结果再过 schema；显示到 HTML/终端前编码。stderr 与 exit reason 保留，但 secrets 先脱敏。

In [ ]:
def capture_output124(text,max_bytes=16):  # 定义本节可复用的核心函数。
    raw=text.encode("utf-8"); clipped=raw[:max_bytes]  # 计算并保存当前步骤的中间状态。
    while True:  # 在终止条件满足前持续推进状态。
        try: decoded=clipped.decode("utf-8"); break  # 尝试执行可能失败的受控操作。
        except UnicodeDecodeError: clipped=clipped[:-1]  # 捕获预期异常并验证失败分支。
    return {"text":decoded,"truncated":len(raw)>len(clipped),"original_bytes":len(raw)}  # 返回当前分支计算出的结果。
cap124=capture_output124("结果:"+"好"*20,16)  # 计算并保存当前步骤的中间状态。
assert cap124["truncated"] and len(cap124["text"].encode())<=16  # 用受控断言验证关键不变量。
assert cap124["original_bytes"]>16  # 用受控断言验证关键不变量。
assert not capture_output124("ok",16)["truncated"]  # 用受控断言验证关键不变量。

## 7. 副作用写 staging，审计 diff 后提交

代码生成的文件、SQL 或补丁先写隔离 staging；检查允许路径、文件数、总大小、二进制类型和危险内容，再由宿主/人工原子提交。取消沙箱只丢弃 staging，不影响生产。不可逆外部动作不在代码 VM 中开放。

In [ ]:
staging124={}  # 计算并保存当前步骤的中间状态。
def stage_write124(path,content,max_files=3,max_bytes=100):  # 定义本节可复用的核心函数。
    if path.startswith("/") or ".." in path.split("/"): return False,"path"  # 按当前条件选择后续控制路径。
    if path not in staging124 and len(staging124)>=max_files: return False,"file_count"  # 按当前条件选择后续控制路径。
    if sum(len(v.encode()) for v in staging124.values())+len(content.encode())>max_bytes: return False,"bytes"  # 按当前条件选择后续控制路径。
    staging124[path]=content; return True,"staged"  # 计算并保存当前步骤的中间状态。
assert stage_write124("out/report.txt","hello")== (True,"staged")  # 用受控断言验证关键不变量。
assert stage_write124("../secret","x")==(False,"path")  # 用受控断言验证关键不变量。
assert staging124=={"out/report.txt":"hello"}  # 用受控断言验证关键不变量。

## 8. 每次执行固定镜像、能力、配额与审计记录

manifest 包含镜像 digest、runtime、CPU/内存/墙钟、网络、mount、能力和输出上限；trace 记录代码 hash、输入 artifact hash、exit reason 和 staged diff，不默认记录敏感内容。教学 AST VM 只能证明应用层约束，不能替代内核隔离。

In [ ]:
source124="max(1,x*2)+3"; manifest124={"schema":1,"runtime":"capability-ast-demo","production_boundary":"microvm_or_container","network":"deny","filesystem":"readonly+staging","cpu_ms":1000,"memory_mb":128,"output_bytes":65536,"source_sha256":hashlib.sha256(source124.encode()).hexdigest()}; digest124=hashlib.sha256(json.dumps(manifest124,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert manifest124["network"]=="deny" and manifest124["memory_mb"]==128  # 用受控断言验证关键不变量。
assert len(manifest124["source_sha256"])==64 and len(digest124)==64  # 用受控断言验证关键不变量。
assert manifest124["production_boundary"]=="microvm_or_container"  # 用受控断言验证关键不变量。

## 面试总结

回答应覆盖：**不在宿主 eval → OS/身份/文件/网络隔离 → AST/能力白名单作为额外层 → gas/数值/时间/内存 → artifact ACL → 网络 broker → 输出 cap → staging diff/人工提交 → 镜像与 trace 版本**。应用级过滤从来不是完整沙箱，边界要由操作系统隔离实现。

延伸阅读：[Firecracker](https://firecracker-microvm.github.io/)、[gVisor Security Model](https://gvisor.dev/docs/architecture_guide/security/)、[OWASP Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)。